In [20]:
import numpy as np
import pandas as pd

In [21]:
votes = pd.read_csv('https://voteview.com/static/data/out/votes/HS119_votes.csv')

In [22]:
votes.head(3).T

,0,1,2
congress,119,119,119
chamber,House,House,House
rollnumber,1,1,1
icpsr,14854,14863,14873
cast_code,1,1,6
prob,99.6,78.9,100.0


# 🐼 skill: filter columns

In [23]:
votes = votes[['chamber', 'rollnumber', 'icpsr', 'cast_code']] # keep columns

# or do this:
# votes = votes.drop(['congress', 'prob'], axis=1) # drop columns

votes

,chamber,rollnumber,icpsr,cast_code
0,House,1,14854,1
1,House,1,14863,1
2,House,1,14873,6
3,House,1,15029,6
4,House,1,15433,6
...,...,...,...,...
383026,Senate,915,42506,1
383027,Senate,915,42507,1
383028,Senate,915,42508,1
383029,Senate,915,49308,6


In [24]:
data1 = [{'id': 1, 'x': 'a'}, {'id': 1, 'x': 'b'}, {'id': 1, 'x': 'c'}]
data2 = [{'id': 1, 'y': 'd'}, {'id': 1, 'y': 'e'}, {'id': 1, 'y': 'f'}]

In [25]:
mergedata = pd.merge(pd.DataFrame(data1),
                     pd.DataFrame(data2),
                     on = 'id')
mergedata

,id,x,y
0,1,a,d
1,1,a,e
2,1,a,f
3,1,b,d
4,1,b,e
5,1,b,f
6,1,c,d
7,1,c,e
8,1,c,f


# 🐼 skill: many-to-many joins (usually we don't want this! but here we do)

In [26]:
vote_compare = pd.merge(votes, votes, 
                        on = ['chamber', 'rollnumber'])
vote_compare

,chamber,rollnumber,icpsr_x,cast_code_x,icpsr_y,cast_code_y
0,House,1,14854,1,14854,1
1,House,1,14854,1,14863,1
2,House,1,14854,1,14873,6
3,House,1,14854,1,15029,6
4,House,1,14854,1,15433,6
...,...,...,...,...,...,...
135162640,Senate,915,49703,1,42506,1
135162641,Senate,915,49703,1,42507,1
135162642,Senate,915,49703,1,42508,1
135162643,Senate,915,49703,1,49308,6


# 🐼 skill: more dropping

In [27]:
vote_compare = vote_compare.drop(['chamber', 'rollnumber'], axis=1)
vote_compare

,icpsr_x,cast_code_x,icpsr_y,cast_code_y
0,14854,1,14854,1
1,14854,1,14863,1
2,14854,1,14873,6
3,14854,1,15029,6
4,14854,1,15433,6
...,...,...,...,...
135162640,49703,1,42506,1
135162641,49703,1,42507,1
135162642,49703,1,42508,1
135162643,49703,1,49308,6


# 🐼 skill: create new column

In [28]:
vote_compare['agree'] = vote_compare['cast_code_x'] == vote_compare['cast_code_y']
vote_compare = vote_compare.drop(['cast_code_x', 'cast_code_y'], axis=1)
vote_compare

,icpsr_x,icpsr_y,agree
0,14854,14854,True
1,14854,14863,True
2,14854,14873,False
3,14854,15029,False
4,14854,15433,False
...,...,...,...
135162640,49703,42506,True
135162641,49703,42507,True
135162642,49703,42508,True
135162643,49703,49308,False


# 🐼 skill: filter rows

In [29]:
vote_compare = vote_compare.query('icpsr_x != icpsr_y')

In [30]:
vote_compare

,icpsr_x,icpsr_y,agree
1,14854,14863,True
2,14854,14873,False
3,14854,15029,False
4,14854,15433,False
5,14854,15448,False
...,...,...,...
135162639,49703,42505,True
135162640,49703,42506,True
135162641,49703,42507,True
135162642,49703,42508,True


# 🐼 skill: rename columns

In [31]:
vote_compare = vote_compare.rename({'icpsr_x': 'icpsr',
                                    'icpsr_y': 'comparison_member'}, axis=1)
vote_compare

,icpsr,comparison_member,agree
1,14854,14863,True
2,14854,14873,False
3,14854,15029,False
4,14854,15433,False
5,14854,15448,False
...,...,...,...
135162639,49703,42505,True
135162640,49703,42506,True
135162641,49703,42507,True
135162642,49703,42508,True


# 🐼 skill: sorting

In [32]:
vote_compare = vote_compare.sort_values(['icpsr', 'comparison_member'], ascending=True)
vote_compare

,icpsr,comparison_member,agree
125985887,14226,14435,False
125995688,14226,14435,False
126005292,14226,14435,False
126015093,14226,14435,False
126024894,14226,14435,False
...,...,...,...
37045188,99912,91980,True
37232677,99912,91980,True
37420166,99912,91980,True
37794279,99912,91980,True


# 🐼 skill: data aggregation

In [33]:
vote_compare = vote_compare.groupby(['icpsr', 'comparison_member']).agg({'agree': 'mean'}).reset_index()
vote_compare

,icpsr,comparison_member,agree
0,14226,14435,0.028415
1,14226,14858,0.077596
2,14226,14871,0.060109
3,14226,14921,0.798907
4,14226,15021,0.123497
...,...,...,...
215349,99912,42505,1.000000
215350,99912,42506,1.000000
215351,99912,49308,0.000000
215352,99912,49703,0.972414


In [39]:
# user supplies 14226, for example
user = '14226'
vote_compare.query(f'icpsr == {user}').sort_values('agree', ascending=True).head(10)

,icpsr,comparison_member,agree
0,14226,14435,0.028415
64,14226,41301,0.030601
37,14226,29147,0.031694
10,14226,20707,0.037158
8,14226,20330,0.046995
52,14226,40908,0.048087
2,14226,14871,0.060109
101,14226,49308,0.061202
29,14226,21706,0.062295
76,14226,41706,0.063388


In [44]:
import os
os.chdir('..')
from contrans import contrans

ct = contrans()

In [ ]:
ct.get_vote_similarity_data()

In [50]:
vote_compare = pd.read_parquet('data/raw/vote_compare.parquet')
vote_compare.query("'MCGUIRE' in bioname")

,bioname,comparison_member,agree


In [55]:
vote_compare[['JOHNSON, Mike' in x for x in vote_compare['bioname']]].sort_values('agree')

,bioname,comparison_member,agree
71098,"JOHNSON, Mike","GRIJALVA, Raúl M.",0.000000
71304,"JOHNSON, Mike","PLASKETT, Stacey E.",0.062500
71290,"JOHNSON, Mike","SHERRILL, Mikie",0.103774
71305,"JOHNSON, Mike","RADEWAGEN, Aumua Amata Coleman",0.125000
71162,"JOHNSON, Mike","SWALWELL, Eric",0.136490
...,...,...,...
71154,"JOHNSON, Mike","FLEISCHMANN, Chuck",0.998081
71358,"JOHNSON, Mike","YAKYM, Rudy, III",0.998081
71125,"JOHNSON, Mike","LATTA, Robert E.",0.998081
71487,"JOHNSON, Mike","TAYLOR, David J.",0.998081
